In [ ]:
import os
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from scipy import stats
from sqlalchemy import URL, create_engine, text
from statsmodels.stats.proportion import proportions_ztest
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor


pio.templates.default = 'plotly_dark'
pio.renderers.default = 'notebook'
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.3f}'.format)

current_dir = Path.cwd().resolve()

ROOT = next(
    (
        path
        for path in [current_dir, *current_dir.parents]
        if (path / "notebooks" / "nba_analysis.ipynb").is_file()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError(
        "Project root was not found. Run the notebook from within the repository."
    )

notebooks_dir = str(ROOT / "notebooks")

if notebooks_dir not in sys.path:
    sys.path.insert(0, notebooks_dir)

import nba_top_players

load_dotenv(ROOT / ".env")

DB_PATH = ROOT / "database" / "data" / "nba_star.db"

if not DB_PATH.is_file():
    raise FileNotFoundError(f"Database file was not found: {DB_PATH}")

In [50]:
engine = create_engine(
    URL.create("sqlite", database=str(DB_PATH))
)

with engine.connect() as connection:
    integrity_result = connection.execute(
        text("PRAGMA integrity_check")
    ).scalars().all()

    table_names = connection.execute(
        text(
            "SELECT name FROM sqlite_master "
            "WHERE type = 'table' ORDER BY name"
        )
    ).scalars().all()

if integrity_result != ["ok"]:
    raise RuntimeError(f"Database integrity check failed: {integrity_result}")

required_tables = {
    "fact_player_stats",
    "dim_player",
    "dim_season",
    "dim_team_scope",
    "dim_player_context",
    "dim_phase",
}

missing_tables = required_tables - set(table_names)

if missing_tables:
    raise RuntimeError(f"Required tables are missing: {sorted(missing_tables)}")

print(f"Database: {DB_PATH}")
print("Integrity check: OK")
print(f"Tables: {len(table_names)}")

Database: G:\بوت کمپ تحلیل داده\New folder\github\nba-data-analysis\database\data\nba_star.db
Integrity check: OK
Tables: 9


In [51]:
# -----------------------                   
# -----df_dim_player-----
# -----------------------
df_dim_player = pd.read_sql('SELECT * from dim_player',engine)

# -------------------------------                   
# -----df_fact_player_salary-----
# -------------------------------
df_fact_player_salary = pd.read_sql('SELECT * from fact_player_salary',engine)
df_fact_player_salary = df_fact_player_salary[df_fact_player_salary['season_key'] != 1]
df_fact_player_salary['season_key']= df_fact_player_salary['season_key'] - 1

# ------------------------------                   
# -----df_fact_player_stats-----
# ------------------------------
df_fact_player_stats = pd.read_sql('SELECT * from fact_player_stats',engine)
df_fact_player_stats = df_fact_player_stats[df_fact_player_stats["is_player_phase_total"] == 1]
df_fact_player_stats = df_fact_player_stats.dropna()
df_fact_player_stats['field_goal_percentage'] = df_fact_player_stats['field_goals']/df_fact_player_stats['field_goal_attempts']
df_fact_player_stats['2P_Rate'] = df_fact_player_stats['two_point_attempts']/df_fact_player_stats['field_goal_attempts']
df_fact_player_stats['2P%'] = df_fact_player_stats['two_point_field_goals']/df_fact_player_stats['two_point_attempts']
df_fact_player_stats['3P_Rate'] = df_fact_player_stats['three_point_attempts']/df_fact_player_stats['field_goal_attempts']
df_fact_player_stats['3P%'] = df_fact_player_stats['three_point_field_goals']/df_fact_player_stats['three_point_attempts']
df_fact_player_stats['free_throws_percentage'] = df_fact_player_stats['free_throws']/df_fact_player_stats['free_throw_attempts']
df_fact_player_stats['awards_source_num'] = (df_fact_player_stats['awards_source_text'].str.strip() != '').astype(int)
df_fact_player_stats.fillna(0,inplace = True)


,stat_key,source_stat_id,player_key,season_key,scope_key,phase_key,context_key,totals_source_rank,advanced_source_rank,is_player_phase_total,awards_source_text,games_played,games_started,minutes_played,field_goals,field_goal_attempts,two_point_field_goals,two_point_attempts,three_point_field_goals,three_point_attempts,...,total_rebound_percentage,assist_percentage,steal_percentage,block_percentage,usage_percentage,offensive_win_shares,defensive_win_shares,win_shares,win_shares_per_48_minutes,offensive_box_plus_minus,defensive_box_plus_minus,box_plus_minus,value_over_replacement_player,field_goal_percentage,2P_Rate,2P%,3P_Rate,3P%,free_throws_percentage,awards_source_num
0,1,1,6,1,12,2,58,127,138,1,,4,2,72,12,21,8,12,4,9,...,4.100,23.400,2.800,0.000,17.400,0.100,0.000,0.200,0.108,0.200,1.400,1.600,0.100,0.571,0.571,0.667,0.429,0.444,0.600,0
1,2,2,9,1,21,2,153,183,169,1,,3,0,25,1,7,0,2,1,5,...,6.300,0.000,1.900,9.800,13.600,-0.100,0.000,0.000,-0.044,-3.100,1.400,-1.700,0.000,0.143,0.286,0.000,0.714,0.200,0.500,0
2,3,3,16,1,23,2,433,130,112,1,,4,3,129,12,25,12,21,0,4,...,12.100,11.600,0.400,3.600,10.500,0.100,0.000,0.100,0.049,-1.800,0.400,-1.300,0.000,0.480,0.840,0.571,0.160,0.000,0.571,0
3,4,4,18,1,23,2,255,115,126,1,,4,0,95,16,49,13,33,3,16,...,8.500,14.100,0.500,3.000,25.700,-0.100,0.000,-0.100,-0.065,-0.800,-2.700,-3.500,0.000,0.327,0.673,0.394,0.327,0.188,0.778,0
4,5,5,22,1,14,2,118,44,19,1,,21,1,510,48,113,31,52,17,61,...,5.600,15.200,2.200,2.300,13.400,0.200,0.600,0.800,0.080,-2.400,1.900,-0.500,0.200,0.425,0.460,0.596,0.540,0.279,0.800,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4681,4682,4682,957,5,4,1,260,225,291,1,,25,23,872,170,376,111,207,59,169,...,8.300,16.800,1.200,0.900,23.800,0.800,0.600,1.500,0.080,0.800,-0.700,0.100,0.500,0.452,0.551,0.536,0.449,0.349,0.854,0
4682,4683,4683,959,5,15,1,147,462,448,1,,7,0,161,17,54,12,37,5,17,...,6.800,22.500,2.100,1.700,17.500,-0.300,0.200,-0.200,-0.046,-6.200,0.300,-5.900,-0.200,0.315,0.685,0.324,0.315,0.294,0.750,0
4683,4684,4684,960,5,8,1,67,351,343,1,,58,0,576,69,149,63,126,6,23,...,12.100,7.000,1.300,5.600,15.400,0.400,0.800,1.200,0.099,-3.800,0.100,-3.700,-0.200,0.463,0.846,0.500,0.154,0.261,0.677,0
4684,4685,4685,962,5,15,1,44,246,256,1,,51,15,1038,150,378,92,189,58,189,...,9.500,11.100,1.700,0.800,19.700,-0.900,1.100,0.200,0.010,-4.000,-0.400,-4.500,-0.600,0.397,0.500,0.487,0.500,0.307,0.827,0


In [52]:
# --------------------------                     
# -----Merge DataFrames-----
# --------------------------
result_1 = pd.merge(
    df_fact_player_stats,
    df_fact_player_salary,
    on = ['player_key','season_key'],
    how = 'inner'
)
result_2 = pd.merge(
    result_1,
    df_dim_player,
    on = 'player_key',
    how = 'inner'
)
print(len(result_2))


2564


In [53]:
# -----------------------------------------                     
# -----Divide data into train and test-----
# -----------------------------------------
result2_sorted =  result_2.sort_values(by = ['player_key','season_key'])
all_data_train =  result2_sorted[result2_sorted['season_key'] != 4]
all_data_test = result2_sorted[result2_sorted['season_key'] == 4]

X_train = all_data_train.iloc[:,[7,8,11,13,14,20,22,23,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48
                                ,49,50,51,52,53,54,60,61]]
X_test = all_data_test.iloc[:,[7,8,11,13,14,20,22,23,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48
                                ,49,50,51,52,53,54,60,61]]
y_train = all_data_train.iloc[:,[56]]
y_test = all_data_test.iloc[:,[56]]

In [54]:
# ----------------------------                     
# -----Data Normalization-----
# ----------------------------
scaler_std = StandardScaler()

X_train_scaled = scaler_std.fit_transform(X_train)

X_test_scaled = scaler_std.transform(X_test)


In [55]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42)
}
results_list = []

print("Starting model comparison...\n")

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    predictions = model.predict(X_test_scaled)
    r2 = r2_score(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    results_list.append({
        "Model": name,
        "R2 Score": r2,
        "MSE": mse
    })
    print(f"✅ {name} completed.")

results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values(by="R2 Score", ascending=False).reset_index(drop=True)
print("\n" + "="*50)
print(results_df.to_string(index=False))
print("\n" + "="*50)

Starting model comparison...

✅ Linear Regression completed.
✅ Ridge Regression completed.


c:\Users\Ali Karimi\.venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


✅ Random Forest completed.
✅ XGBoost completed.

            Model  R2 Score                MSE
    Random Forest     0.581 61626379949832.656
          XGBoost     0.577 62141059039232.000
 Ridge Regression     0.552 65820502203174.164
Linear Regression     0.551 65890746451513.648

